# LeetCode 890 — Find and Replace Pattern bằng Python

Link đề: https://leetcode.com/problems/find-and-replace-pattern/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), String (chuỗi)

Skeleton LeetCode:

```python
class Solution:
    def findAndReplacePattern(self, words: List[str], pattern: str) -> List[str]:
        ...
```

Ghi chú: một từ "khớp" với `pattern` khi có thể **thay mỗi chữ cái của `pattern` bằng một chữ cái khác nhau** (một **song ánh** giữa các chữ cái) để thu được đúng từ đó. Nói cách khác, hai chuỗi có **cùng "hình dạng lặp lại"**: vị trí nào của `pattern` giống nhau thì vị trí đó của từ cũng giống nhau, và ngược lại. Đề cho phép trả kết quả theo **thứ tự bất kỳ**, nên khi kiểm thử ta so sánh sau khi sắp xếp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm các từ `words[i]` mà tồn tại một hoán vị `p` của bảng chữ cái sao cho thay mỗi chữ `x` của `pattern` bằng `p(x)` thì được đúng từ đó.
2. Nắm các công cụ Python cần thiết: `dict` và `set`, `zip(a, b)` để duyệt song song hai chuỗi, `dict.setdefault`, `itertools.permutations`, và so sánh danh sách sau khi `sorted`.
3. Hiểu nguyên lý **song ánh (bijection)**: ánh xạ phải **nhất quán** (một chữ chỉ có một ảnh) và **đơn ánh** (hai chữ khác nhau không có cùng ảnh), nên cần kiểm tra **cả hai chiều**.
4. Đi qua 4 cách: thử mọi hoán vị bảng chữ cái (brute, chỉ chạy được trên bảng chữ cái nhỏ), hai bảng ánh xạ xuôi/ngược, **chuẩn hóa chuỗi** (đánh số theo lần xuất hiện đầu) và so sánh số lượng `set(zip(...))`; đối chiếu bằng oracle độc lập.
5. Tránh các lỗi kinh điển: chỉ kiểm tra một chiều ánh xạ, chỉ so sánh số chữ khác nhau hoặc phân bố tần suất, dùng chung bảng ánh xạ giữa các từ, và quên rằng hai chữ khác nhau không được trùng ảnh.

## 2. Đề bài & ví dụ

Cho danh sách chuỗi `words` và một chuỗi `pattern`. Trả về danh sách các `words[i]` **khớp** với `pattern`. Có thể trả về theo **thứ tự bất kỳ**.

Một từ khớp với `pattern` nếu tồn tại một hoán vị các chữ cái `p` sao cho sau khi thay mỗi chữ `x` của `pattern` bằng `p(x)` ta được đúng từ đó.

Nhắc lại: hoán vị chữ cái là một **song ánh** từ chữ cái sang chữ cái: mỗi chữ ánh xạ đến một chữ và không hai chữ nào ánh xạ đến cùng một chữ.

**Ví dụ 1**

- Input: `words = ["abc","deq","mee","aqq","dkd","ccc"]`, `pattern = "abb"`
- Output: `["mee","aqq"]`
- Giải thích: `"mee"` khớp vì có hoán vị `{a -> m, b -> e, ...}`; `"ccc"` không khớp vì `a` và `b` cùng ánh xạ đến `c`, không phải hoán vị.

**Ví dụ 2**

- Input: `words = ["a","b","c"]`, `pattern = "a"`
- Output: `["a","b","c"]`

**Ràng buộc:**

- `1 <= pattern.length <= 20`
- `1 <= words.length <= 50`
- `words[i].length == pattern.length`
- `pattern` và `words[i]` gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
from itertools import permutations
from collections import Counter
import random, string, time

examples = [
    (["abc", "deq", "mee", "aqq", "dkd", "ccc"], "abb", ["mee", "aqq"]),
    (["a", "b", "c"], "a", ["a", "b", "c"]),
]
for words, pattern, expected in examples:
    print(f"words={words} pattern={pattern!r} kỳ vọng={expected}")

words=['abc', 'deq', 'mee', 'aqq', 'dkd', 'ccc'] pattern='abb' kỳ vọng=['mee', 'aqq']
words=['a', 'b', 'c'] pattern='a' kỳ vọng=['a', 'b', 'c']


## 3. Kiến thức nền

### 3.1. Song ánh giữa chữ cái

Một ánh xạ `f` từ chữ của `pattern` sang chữ của từ cần thỏa hai điều kiện:

1. **Hàm (nhất quán):** cùng một chữ trong `pattern` luôn cho cùng một chữ trong từ.
2. **Đơn ánh (không trùng ảnh):** hai chữ **khác nhau** trong `pattern` cho hai chữ **khác nhau** trong từ.

Điều kiện 2 bị bỏ sót nhiều nhất: ví dụ `pattern = "abb"` và từ `"ccc"` thỏa điều kiện 1 (`a -> c`, `b -> c`) nhưng vi phạm điều kiện 2 vì hai chữ `a`, `b` cùng ảnh `c`.

In [2]:
pattern, word = "abb", "ccc"
f = {}
for x, y in zip(pattern, word):
    f.setdefault(x, set()).add(y)
print("ánh xạ xuôi:", f)
print("nhất quán (mỗi chữ chỉ một ảnh)?", all(len(v) == 1 for v in f.values()))
images = [next(iter(v)) for v in f.values()]
print("đơn ánh (ảnh đôi một khác nhau)?", len(set(images)) == len(images))

ánh xạ xuôi: {'a': {'c'}, 'b': {'c'}}
nhất quán (mỗi chữ chỉ một ảnh)? True
đơn ánh (ảnh đôi một khác nhau)? False


### 3.2. `zip` để duyệt song song và `set` các cặp

`zip(pattern, word)` ghép từng cặp vị trí tương ứng. `set(zip(pattern, word))` là tập các cặp **khác nhau**: nếu ánh xạ là song ánh thì số cặp khác nhau bằng số chữ khác nhau của `pattern` **và** bằng số chữ khác nhau của từ.

In [3]:
for pattern, word in (("abb", "mee"), ("abb", "ccc"), ("abb", "aab"), ("ab", "ba")):
    pairs = set(zip(pattern, word))
    print(f"{pattern} vs {word}: cặp {sorted(pairs)} | số chữ pattern {len(set(pattern))} | số chữ từ {len(set(word))} | số cặp {len(pairs)}")

abb vs mee: cặp [('a', 'm'), ('b', 'e')] | số chữ pattern 2 | số chữ từ 2 | số cặp 2
abb vs ccc: cặp [('a', 'c'), ('b', 'c')] | số chữ pattern 2 | số chữ từ 1 | số cặp 2
abb vs aab: cặp [('a', 'a'), ('b', 'a'), ('b', 'b')] | số chữ pattern 2 | số chữ từ 2 | số cặp 3
ab vs ba: cặp [('a', 'b'), ('b', 'a')] | số chữ pattern 2 | số chữ từ 2 | số cặp 2


### 3.3. Chuẩn hóa chuỗi: đánh số theo lần xuất hiện đầu

Thay mỗi chữ bằng **số thứ tự xuất hiện đầu tiên** của nó: `"abb" -> (0, 1, 1)`, `"mee" -> (0, 1, 1)`, `"ccc" -> (0, 0, 0)`. Hai chuỗi khớp nhau (theo nghĩa song ánh chữ cái) khi và chỉ khi dạng chuẩn hóa **bằng nhau**.

In [4]:
def canon(s: str) -> tuple:
    first = {}
    return tuple(first.setdefault(ch, len(first)) for ch in s)

for s in ("abb", "mee", "ccc", "abc", "deq", "dkd"):
    print(f"{s}: {canon(s)}")

abb: (0, 1, 1)
mee: (0, 1, 1)
ccc: (0, 0, 0)
abc: (0, 1, 2)
deq: (0, 1, 2)
dkd: (0, 1, 0)


### 3.4. So sánh danh sách không quan tâm thứ tự

Đề cho phép trả theo thứ tự bất kỳ nên khi kiểm thử ta so sánh `sorted(kết quả)` với `sorted(kỳ vọng)` (với từ trùng lặp, `sorted` giữ đúng số lần xuất hiện).

In [5]:
a, b = ["mee", "aqq"], ["aqq", "mee"]
print(a == b, sorted(a) == sorted(b))

False True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `pattern = "abb"`. Hình dạng lặp lại là `(0, 1, 1)`: hai vị trí cuối **giống nhau** và khác vị trí đầu.

| Từ | Hình dạng | Khớp? | Lý do |
|---|---|---|---|
| `abc` | `(0,1,2)` | không | `b` và `c` khác nhau nhưng `pattern` đòi bằng nhau |
| `deq` | `(0,1,2)` | không | như trên |
| `mee` | `(0,1,1)` | **có** | `a -> m`, `b -> e` |
| `aqq` | `(0,1,1)` | **có** | `a -> a`, `b -> q` |
| `dkd` | `(0,1,0)` | không | vị trí đầu và cuối bằng nhau nhưng `pattern` đòi vị trí đầu khác vị trí cuối |
| `ccc` | `(0,0,0)` | không | `a` và `b` đều thành `c` |

Vậy kết quả là `["mee", "aqq"]`.

## 5. Cách 1 — Brute Force: thử mọi hoán vị của bảng chữ cái

**Ý tưởng.** Đúng định nghĩa: một từ khớp nếu **tồn tại** hoán vị `p` của bảng chữ cái sao cho `p(pattern[i]) == word[i]` với mọi `i`. Ta thử **mọi** hoán vị. Với 26 chữ cái có `26!` hoán vị nên không thể chạy thật; ta chạy trên bảng chữ cái **nhỏ** (ví dụ 5-7 chữ cái) để làm **oracle định nghĩa**.

In [6]:
def match_by_permutation(word: str, pattern: str, alphabet: str) -> bool:
    for perm in permutations(alphabet):
        p = dict(zip(alphabet, perm))
        if all(p[x] == y for x, y in zip(pattern, word)):
            return True
    return False

def find_brute(words: List[str], pattern: str, alphabet: str) -> List[str]:
    return [w for w in words if match_by_permutation(w, pattern, alphabet)]

# ví dụ 1 chỉ dùng các chữ a b c d e k m q; ta cần bảng chữ cái đủ lớn chứa mọi chữ xuất hiện
alphabet = "abcdekmq"
got = find_brute(examples[0][0], examples[0][1], alphabet)
print(got)
assert sorted(got) == sorted(examples[0][2])

['mee', 'aqq']


**Độ phức tạp và nút thắt.** `O(k! * L)` cho mỗi từ với `k` là cỡ bảng chữ cái: với `k = 26` là không thể. Hãy xem thời gian tăng theo `k`:

In [7]:
for k in (5, 6, 7, 8, 9):
    alphabet = string.ascii_lowercase[:k]
    t0 = time.perf_counter()
    r = match_by_permutation("a" * 4, "abcd", alphabet)         # không khớp: phải thử hết mọi hoán vị
    print(f"k={k}: {1000 * (time.perf_counter() - t0):9.1f} ms (kết quả {r})")

k=5:       0.1 ms (kết quả False)
k=6:       0.6 ms (kết quả False)
k=7:       6.5 ms (kết quả False)
k=8:      58.6 ms (kết quả False)


k=9:     361.9 ms (kết quả False)


Thời gian tăng gần **gấp `k` lần mỗi chữ cái thêm vào**. Nút thắt: ta thử cả những hoán vị **không liên quan** tới từ đang xét, trong khi ánh xạ cần thiết bị **ép buộc** hoàn toàn bởi hai chuỗi: `pattern[i] -> word[i]`.

## 6. Ý tưởng thuật toán / bước đột phá

**Ánh xạ bị ép buộc bởi chính hai chuỗi.** Nếu từ khớp thì hoán vị `p` phải thỏa `p(pattern[i]) = word[i]` với mọi `i`; các chữ không xuất hiện trong `pattern` được gán tùy ý (hoán vị luôn mở rộng được nếu phần đã gán là **đơn ánh**). Do đó chỉ cần kiểm tra trực tiếp, trong `O(L)`:

1. mỗi chữ của `pattern` ứng với **đúng một** chữ của từ (hàm);
2. mỗi chữ của từ ứng với **đúng một** chữ của `pattern` (đơn ánh).

| Cách | Ý tưởng |
|---|---|
| 2 | Hai bảng băm: ánh xạ xuôi `pattern -> word` và ngược `word -> pattern` |
| 3 | **Chuẩn hóa**: mỗi chuỗi thành dãy số theo lần xuất hiện đầu; so bằng dãy |
| 4 | Đếm: `len(set(pattern)) == len(set(word)) == len(set(zip(pattern, word)))` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hai bảng ánh xạ xuôi và ngược

**Giải thích từng đoạn.**

- `m1[x]` là chữ của từ ứng với chữ `x` của `pattern`; `m2[y]` là chữ của `pattern` ứng với chữ `y` của từ.
- Với mỗi vị trí `(x, y)`:
  - nếu `x` đã có ảnh `m1[x]` mà khác `y` ⇒ không nhất quán ⇒ sai;
  - nếu `y` đã có tạo ảnh ngược `m2[y]` mà khác `x` ⇒ hai chữ khác nhau cùng ảnh ⇒ sai;
  - ngược lại ghi `m1[x] = y`, `m2[y] = x`.
- Bảng được tạo **lại cho từng từ** (không dùng chung giữa các từ).

In [8]:
def match_two_maps(word: str, pattern: str) -> bool:
    m1, m2 = {}, {}
    for x, y in zip(pattern, word):
        if m1.get(x, y) != y or m2.get(y, x) != x:
            return False
        m1[x], m2[y] = y, x
    return True

def find_two_maps(words: List[str], pattern: str) -> List[str]:
    return [w for w in words if match_two_maps(w, pattern)]

for words, pattern, expected in examples:
    got = find_two_maps(words, pattern)
    print(words, pattern, "->", got, "OK" if sorted(got) == sorted(expected) else "SAI")
    assert sorted(got) == sorted(expected)

['abc', 'deq', 'mee', 'aqq', 'dkd', 'ccc'] abb -> ['mee', 'aqq'] OK
['a', 'b', 'c'] a -> ['a', 'b', 'c'] OK


**Code nộp LeetCode (Cách 2):**

In [9]:
class Solution:
    def findAndReplacePattern(self, words: List[str], pattern: str) -> List[str]:
        def match(word: str) -> bool:
            m1, m2 = {}, {}
            for x, y in zip(pattern, word):
                if m1.get(x, y) != y or m2.get(y, x) != x:
                    return False
                m1[x], m2[y] = y, x
            return True

        return [w for w in words if match(w)]

sol_maps = Solution().findAndReplacePattern
for words, pattern, expected in examples:
    assert sorted(sol_maps(words, pattern)) == sorted(expected)
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(w * L)` thời gian với `w` là số từ và `L` độ dài; bộ nhớ `O(min(L, 26))` cho hai bảng.

---

### 7.2. Cách 3 — Chuẩn hóa chuỗi (đánh số theo lần xuất hiện đầu)

**Ý tưởng khác.** Biến mỗi chuỗi thành **dãy số**: chữ gặp lần đầu nhận số kế tiếp `0, 1, 2, ...`; chữ đã gặp giữ số cũ. Hai chuỗi khớp song ánh khi và chỉ khi dãy số bằng nhau. Chuẩn hóa `pattern` một lần, rồi so với dạng chuẩn hóa của từng từ.

**Giải thích từng đoạn.** `m.setdefault(c, len(m))` trả số của `c` nếu đã có, ngược lại gán `len(m)` (số chữ khác nhau đã gặp) rồi trả về nó.

In [10]:
def find_canon(words: List[str], pattern: str) -> List[str]:
    def f(s: str) -> List[int]:
        m = {}
        return [m.setdefault(c, len(m)) for c in s]

    target = f(pattern)
    return [w for w in words if f(w) == target]

for words, pattern, expected in examples:
    got = find_canon(words, pattern)
    print(words, pattern, "->", got, "OK" if sorted(got) == sorted(expected) else "SAI")
    assert sorted(got) == sorted(expected)

['abc', 'deq', 'mee', 'aqq', 'dkd', 'ccc'] abb -> ['mee', 'aqq'] OK
['a', 'b', 'c'] a -> ['a', 'b', 'c'] OK


**Code nộp LeetCode (Cách 3):**

In [11]:
class Solution:
    def findAndReplacePattern(self, words: List[str], pattern: str) -> List[str]:
        def f(s: str) -> List[int]:
            m = {}
            return [m.setdefault(c, len(m)) for c in s]

        target = f(pattern)
        return [w for w in words if f(w) == target]

sol_canon = Solution().findAndReplacePattern
for words, pattern, expected in examples:
    assert sorted(sol_canon(words, pattern)) == sorted(expected)
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(w * L)` thời gian; `O(L)` bộ nhớ phụ cho mỗi lần chuẩn hóa.

---

### 7.3. Cách 4 — So số lượng `set(zip(pattern, word))`

**Ý tưởng khác.** Với từ có cùng độ dài, ánh xạ `pattern[i] -> word[i]` là song ánh khi và chỉ khi **ba số bằng nhau**: số chữ khác nhau của `pattern`, số chữ khác nhau của từ, và số **cặp** khác nhau `(pattern[i], word[i])`.

- Nếu số cặp khác nhau bằng số chữ khác nhau của `pattern` thì mỗi chữ của `pattern` chỉ có một ảnh (hàm).
- Nếu số cặp khác nhau bằng số chữ khác nhau của từ thì mỗi chữ của từ chỉ có một nguồn (đơn ánh).

Gọn một dòng.

In [12]:
def match_zip(word: str, pattern: str) -> bool:
    return len(set(pattern)) == len(set(word)) == len(set(zip(pattern, word)))

def find_zip(words: List[str], pattern: str) -> List[str]:
    return [w for w in words if match_zip(w, pattern)]

for words, pattern, expected in examples:
    got = find_zip(words, pattern)
    print(words, pattern, "->", got, "OK" if sorted(got) == sorted(expected) else "SAI")
    assert sorted(got) == sorted(expected)

['abc', 'deq', 'mee', 'aqq', 'dkd', 'ccc'] abb -> ['mee', 'aqq'] OK
['a', 'b', 'c'] a -> ['a', 'b', 'c'] OK


**Code nộp LeetCode (Cách 4):**

In [13]:
class Solution:
    def findAndReplacePattern(self, words: List[str], pattern: str) -> List[str]:
        n = len(set(pattern))
        return [w for w in words if len(set(w)) == n == len(set(zip(pattern, w)))]

sol_zip = Solution().findAndReplacePattern
for words, pattern, expected in examples:
    assert sorted(sol_zip(words, pattern)) == sorted(expected)
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(w * L)` thời gian; `O(L)` bộ nhớ phụ cho các `set`. Chú ý: hoạt động đúng vì hai chuỗi **cùng độ dài** (ràng buộc của đề).

## 8. TRACE từng bước

### 8.1. Cách 2 (hai bảng) cho `pattern = "abb"` với từ `"mee"` và `"ccc"`

In [14]:
def trace_two_maps(word, pattern):
    m1, m2 = {}, {}
    print(f"từ {word!r} so với pattern {pattern!r}")
    for x, y in zip(pattern, word):
        if m1.get(x, y) != y:
            print(f"   {x}->{y}: nhưng {x} đã ánh xạ tới {m1[x]} => SAI (không nhất quán)")
            return False
        if m2.get(y, x) != x:
            print(f"   {x}->{y}: nhưng {y} đã là ảnh của {m2[y]} => SAI (hai chữ cùng ảnh)")
            return False
        m1[x], m2[y] = y, x
        print(f"   {x}->{y}: ghi nhận, xuôi={m1} ngược={m2}")
    print("   => khớp")
    return True

assert trace_two_maps("mee", "abb") is True
assert trace_two_maps("ccc", "abb") is False

từ 'mee' so với pattern 'abb'
   a->m: ghi nhận, xuôi={'a': 'm'} ngược={'m': 'a'}
   b->e: ghi nhận, xuôi={'a': 'm', 'b': 'e'} ngược={'m': 'a', 'e': 'b'}
   b->e: ghi nhận, xuôi={'a': 'm', 'b': 'e'} ngược={'m': 'a', 'e': 'b'}
   => khớp
từ 'ccc' so với pattern 'abb'
   a->c: ghi nhận, xuôi={'a': 'c'} ngược={'c': 'a'}
   b->c: nhưng c đã là ảnh của a => SAI (hai chữ cùng ảnh)


### 8.2. Cách 3 (chuẩn hóa) cho cả danh sách ví dụ 1

In [15]:
def trace_canon(words, pattern):
    f = lambda s: canon(s)
    target = f(pattern)
    print(f"pattern {pattern!r} -> {target}")
    for w in words:
        print(f"   {w!r} -> {f(w)} {'KHỚP' if f(w) == target else ''}")

trace_canon(*examples[0][:2])

pattern 'abb' -> (0, 1, 1)
   'abc' -> (0, 1, 2) 
   'deq' -> (0, 1, 2) 
   'mee' -> (0, 1, 1) KHỚP
   'aqq' -> (0, 1, 1) KHỚP
   'dkd' -> (0, 1, 0) 
   'ccc' -> (0, 0, 0) 


### 8.3. Cách 4 (đếm) cho vài từ

In [16]:
pattern = "abb"
for w in ["abc", "mee", "dkd", "ccc"]:
    a, b, c = len(set(pattern)), len(set(w)), len(set(zip(pattern, w)))
    print(f"{w}: #chữ pattern={a}, #chữ từ={b}, #cặp={c} -> {a == b == c}")
assert [w for w in ["abc", "mee", "dkd", "ccc"] if match_zip(w, pattern)] == ["mee"]

abc: #chữ pattern=2, #chữ từ=3, #cặp=3 -> False
mee: #chữ pattern=2, #chữ từ=2, #cặp=2 -> True
dkd: #chữ pattern=2, #chữ từ=2, #cặp=3 -> False
ccc: #chữ pattern=2, #chữ từ=1, #cặp=2 -> False


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (chuẩn hóa theo lần xuất hiện đầu): trực quan ("hai chuỗi có cùng hình dạng lặp lại"), `O(w * L)`, và mở rộng tự nhiên sang bài nhóm các chuỗi đẳng cấu (Group Isomorphic Strings).

```python
class Solution:
    def findAndReplacePattern(self, words: List[str], pattern: str) -> List[str]:
        def f(s: str) -> List[int]:
            m = {}
            return [m.setdefault(c, len(m)) for c in s]

        target = f(pattern)
        return [w for w in words if f(w) == target]
```

In [17]:
class Solution:
    def findAndReplacePattern(self, words: List[str], pattern: str) -> List[str]:
        def f(s: str) -> List[int]:
            m = {}
            return [m.setdefault(c, len(m)) for c in s]

        target = f(pattern)
        return [w for w in words if f(w) == target]

sol = Solution()
for words, pattern, expected in examples:
    got = sol.findAndReplacePattern(words, pattern)
    print(words, pattern, "->", got, "OK" if sorted(got) == sorted(expected) else "SAI")
    assert sorted(got) == sorted(expected)

['abc', 'deq', 'mee', 'aqq', 'dkd', 'ccc'] abb -> ['mee', 'aqq'] OK
['a', 'b', 'c'] a -> ['a', 'b', 'c'] OK


## 10. Vì sao đúng

**(a) Từ khớp ⇔ ánh xạ `pattern[i] -> word[i]` là một song ánh giữa tập chữ của `pattern` và tập chữ của từ.**

- (⇒) Nếu có hoán vị `p` với `p(pattern[i]) = word[i]` thì ánh xạ `x -> p(x)` hạn chế lên các chữ của `pattern` là hàm (vì `p` là hàm) và đơn ánh (vì `p` là đơn ánh), nên các cặp `(pattern[i], word[i])` định nghĩa một song ánh giữa hai tập chữ.
- (⇐) Nếu các cặp định nghĩa một song ánh `g` giữa tập chữ `A` của `pattern` và tập chữ `B` của từ (`|A| = |B|`), ta mở rộng `g` thành hoán vị của cả bảng chữ cái bằng cách ghép tùy ý các chữ còn lại của bảng chữ cái ngoài `A` với các chữ còn lại ngoài `B` (hai phần bù có cùng cỡ). Đó là hoán vị `p` thỏa điều kiện.

**(b) Cách 2** kiểm tra trực tiếp tính nhất quán (`m1`) và đơn ánh (`m2`) từng vị trí, nên bằng (a).

**(c) Cách 3 (chuẩn hóa).** Dạng chuẩn hóa `canon(s)` ghi lại với mỗi vị trí `i`: chỉ số lần xuất hiện đầu của chữ `s[i]`. Hai chuỗi `s`, `t` cùng độ dài có `canon(s) = canon(t)` ⇔ với mọi `i, j`: `s[i] = s[j]` ⇔ `t[i] = t[j]`. Điều này chính là "các cặp `(s[i], t[i])` là song ánh" vì: `s[i] = s[j] ⇒ t[i] = t[j]` là tính hàm; `t[i] = t[j] ⇒ s[i] = s[j]` là tính đơn ánh.

**(d) Cách 4.** Gọi `P` là tập cặp `(pattern[i], word[i])`. Có `|P| >= |set(pattern)|` và `|P| >= |set(word)|` (mỗi chữ xuất hiện trong ít nhất một cặp). Đẳng thức `|P| = |set(pattern)|` xảy ra khi và chỉ khi mỗi chữ của `pattern` chỉ xuất hiện trong **một** cặp (hàm); `|P| = |set(word)|` khi và chỉ khi mỗi chữ của từ chỉ xuất hiện trong một cặp (đơn ánh). Vậy ba số bằng nhau ⇔ song ánh.

**Kiểm chứng bằng máy.** Oracle **độc lập**: (1) với bảng chữ cái nhỏ (4-6 chữ cái), duyệt **mọi hoán vị** của bảng chữ cái (đúng định nghĩa); (2) với bảng chữ cái đầy đủ, kiểm tra quan hệ cặp `(pattern[i], word[i])` bằng **hai bảng tập hợp** (mỗi chữ có đúng một ảnh và đúng một tiền ảnh), cách cài đặt khác hẳn bốn lời giải. Kết quả được so sánh sau khi sắp xếp.

In [18]:
def oracle_relation(word: str, pattern: str) -> bool:
    """Quan hệ cặp (pattern[i], word[i]) phải là song ánh: mỗi chữ có đúng một ảnh và đúng một tiền ảnh."""
    images, preimages = {}, {}
    for x, y in zip(pattern, word):
        images.setdefault(x, set()).add(y)
        preimages.setdefault(y, set()).add(x)
    return all(len(v) == 1 for v in images.values()) and all(len(v) == 1 for v in preimages.values())

def oracle_find(words: List[str], pattern: str) -> List[str]:
    return sorted(w for w in words if oracle_relation(w, pattern))

# hai oracle khớp nhau trên bảng chữ cái nhỏ
rng = random.Random(890)
for _ in range(600):
    k = rng.randint(2, 5)
    alphabet = string.ascii_lowercase[:k]
    L = rng.randint(1, 6)
    pattern = "".join(rng.choice(alphabet) for _ in range(L))
    word = "".join(rng.choice(alphabet) for _ in range(L))
    assert match_by_permutation(word, pattern, alphabet) == oracle_relation(word, pattern), (word, pattern)
print("oracle hoán vị (định nghĩa) và oracle quan hệ cặp khớp nhau trên 600 cặp ngẫu nhiên")

oracle hoán vị (định nghĩa) và oracle quan hệ cặp khớp nhau trên 600 cặp ngẫu nhiên


## 11. Time / Space Complexity

Gọi `w` là số từ, `L` độ dài mỗi chuỗi, `k` cỡ bảng chữ cái.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị | `O(w * k! * L)` | `O(k)` |
| 2. Hai bảng ánh xạ | `O(w * L)` | `O(min(L, 26))` |
| 3. Chuẩn hóa | `O(w * L)` | `O(L)` |
| 4. `set(zip(...))` | `O(w * L)` | `O(L)` |

Đo thử với đầu vào lớn hơn nhiều so với giới hạn đề (để thấy xu hướng tuyến tính):

In [19]:
rng = random.Random(11)
pattern = "".join(rng.choice("abcdefghij") for _ in range(20))
words = ["".join(rng.choice("abcdefghij") for _ in range(20)) for _ in range(20000)]
# thêm nhiều từ chắc chắn khớp (đổi chữ theo một hoán vị ngẫu nhiên)
for _ in range(2000):
    letters = list(string.ascii_lowercase)
    rng.shuffle(letters)
    p = dict(zip(string.ascii_lowercase, letters))
    words.append("".join(p[c] for c in pattern))
ref = oracle_find(words, pattern)
for name, fn in (("Cách 2 hai bảng", sol_maps), ("Cách 3 chuẩn hóa", sol_canon), ("Cách 4 set(zip)", sol_zip), ("nộp", Solution().findAndReplacePattern)):
    t0 = time.perf_counter()
    got = fn(words, pattern)
    dt = (time.perf_counter() - t0) * 1000
    assert sorted(got) == ref, name
    print(f"{name:18} w={len(words)}: {dt:8.1f} ms, {len(got)} từ khớp")

Cách 2 hai bảng    w=22000:     13.5 ms, 2000 từ khớp
Cách 3 chuẩn hóa   w=22000:     25.7 ms, 2000 từ khớp
Cách 4 set(zip)    w=22000:     19.8 ms, 2000 từ khớp
nộp                w=22000:     25.4 ms, 2000 từ khớp


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với hai oracle, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất. Mọi so sánh dùng `sorted` vì thứ tự kết quả không quan trọng.

In [20]:
def solutions_list():
    return {
        "maps": find_two_maps,
        "canon": find_canon,
        "zip": find_zip,
        "sol_maps": sol_maps,
        "sol_canon": sol_canon,
        "sol_zip": sol_zip,
        "final": Solution().findAndReplacePattern,
    }

def run_all(words, pattern, expected=None, alphabet=None):
    ref = oracle_find(words, pattern)
    if alphabet is not None:
        assert sorted(find_brute(words, pattern, alphabet)) == ref, ("oracle hoán vị lệch", words, pattern)
    for name, fn in solutions_list().items():
        assert sorted(fn(list(words), pattern)) == ref, (name, words, pattern)
    if expected is not None:
        assert ref == sorted(expected), ("kỳ vọng sai", words, pattern, expected, ref)
    return ref

# 12.1 ví dụ đề
for words, pattern, expected in examples:
    run_all(words, pattern, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (["a"], "a", ["a"]),                                   # độ dài 1
    (["z"], "a", ["z"]),
    (["aaa", "bbb", "abc", "aab", "baa"], "xxx", ["aaa", "bbb"]),        # pattern toàn giống nhau
    (["abc", "xyz", "xxy", "xyx", "cba"], "pqr", ["abc", "xyz", "cba"]), # pattern toàn khác nhau
    (["ba", "ab", "aa", "bb"], "ab", ["ba", "ab"]),                       # hoán đổi hai chữ
    (["xyyx", "xyxy", "aabb", "abba", "abcd"], "abba", ["xyyx", "abba"]),
    (["abc", "abc", "xyz"], "abc", ["abc", "abc", "xyz"]),                # từ trùng lặp vẫn giữ
    (["zzzz"], "abcd", []),                                               # không từ nào khớp
    (["abcdefghijklmnopqrst", "tsrqponmlkjihgfedcba"], "abcdefghijklmnopqrst", ["abcdefghijklmnopqrst", "tsrqponmlkjihgfedcba"]),  # độ dài tối đa 20, toàn khác nhau
    (["a" * 20, "b" * 20, "ab" * 10], "c" * 20, ["a" * 20, "b" * 20]),     # độ dài 20 toàn giống nhau
]
for words, pattern, expected in edge:
    run_all(words, pattern, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 10 ca


In [21]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_forward_only(words, pattern):                # chỉ kiểm tra ánh xạ xuôi (không kiểm tra đơn ánh)
    res = []
    for w in words:
        m = {}
        if all(m.setdefault(x, y) == y for x, y in zip(pattern, w)):
            res.append(w)
    return res

def wrong_distinct_count(words, pattern):              # chỉ so số chữ khác nhau
    return [w for w in words if len(set(w)) == len(set(pattern))]

def wrong_sorted_counts(words, pattern):               # so phân bố tần suất (đa tập số lần xuất hiện)
    target = sorted(Counter(pattern).values())
    return [w for w in words if sorted(Counter(w).values()) == target]

def wrong_shared_map(words, pattern):                  # dùng chung bảng ánh xạ giữa các từ
    m1, m2 = {}, {}
    res = []
    for w in words:
        ok = True
        for x, y in zip(pattern, w):
            if m1.get(x, y) != y or m2.get(y, x) != x:
                ok = False
                break
            m1[x], m2[y] = y, x
        if ok:
            res.append(w)
    return res

def find_counter(wrong, tries=20000, seed=4):
    rng = random.Random(seed)
    for _ in range(tries):
        L = rng.randint(2, 5)
        alphabet = "abcd"
        pattern = "".join(rng.choice(alphabet) for _ in range(L))
        words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 4))]
        if sorted(wrong(words, pattern)) != oracle_find(words, pattern):
            return words, pattern
    return None

counters = {}
for name, fn in (("chỉ ánh xạ xuôi", wrong_forward_only), ("chỉ đếm số chữ khác nhau", wrong_distinct_count),
                 ("so phân bố tần suất", wrong_sorted_counts), ("dùng chung bảng giữa các từ", wrong_shared_map)):
    c = find_counter(fn)
    assert c is not None, name
    counters[name] = c
    words, pattern = c
    print(f"{name:28} words={words} pattern={pattern!r}: sai -> {sorted(fn(words, pattern))}, đúng -> {oracle_find(words, pattern)}")
    run_all(words, pattern)
print("12.3 phản ví dụ: OK")

chỉ ánh xạ xuôi              words=['baa', 'adc', 'abc', 'cba'] pattern='cad': sai -> ['abc', 'adc', 'baa', 'cba'], đúng -> ['abc', 'adc', 'cba']
chỉ đếm số chữ khác nhau     words=['dbbc', 'caaa', 'dcdc', 'bbad'] pattern='bddc': sai -> ['bbad', 'dbbc'], đúng -> ['dbbc']
so phân bố tần suất          words=['dbbc', 'caaa', 'dcdc', 'bbad'] pattern='bddc': sai -> ['bbad', 'dbbc'], đúng -> ['dbbc']
dùng chung bảng giữa các từ  words=['baa', 'adc', 'abc', 'cba'] pattern='cad': sai -> [], đúng -> ['abc', 'adc', 'cba']
12.3 phản ví dụ: OK


In [22]:
# 12.4 ngẫu nhiên nhỏ trên bảng chữ cái 4-5 chữ cái: so với oracle hoán vị (định nghĩa) và oracle quan hệ
rng = random.Random(4890)
hits = 0
for _ in range(800):
    k = rng.randint(2, 5)
    alphabet = string.ascii_lowercase[:k]
    L = rng.randint(1, 6)
    pattern = "".join(rng.choice(alphabet) for _ in range(L))
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 8))]
    hits += len(run_all(words, pattern, None, alphabet=alphabet))
print(f"12.4 ngẫu nhiên bảng chữ cái nhỏ (800 ca): OK, tổng {hits} từ khớp")

# 12.5 ngẫu nhiên đúng ràng buộc: pattern dài tới 20, tới 50 từ, 26 chữ cái; có cài sẵn các từ chắc chắn khớp
rng = random.Random(5890)
for _ in range(500):
    L = rng.randint(1, 20)
    sub = rng.randint(1, 26)
    letters = string.ascii_lowercase[:sub]
    pattern = "".join(rng.choice(letters) for _ in range(L))
    words = []
    for _ in range(rng.randint(1, 50)):
        if rng.random() < 0.4:
            perm = list(string.ascii_lowercase)
            rng.shuffle(perm)
            p = dict(zip(string.ascii_lowercase, perm))
            words.append("".join(p[c] for c in pattern))               # chắc chắn khớp
        else:
            words.append("".join(rng.choice(string.ascii_lowercase[:rng.randint(1, 26)]) for _ in range(L)))
    run_all(words, pattern)
print("12.5 ngẫu nhiên đúng ràng buộc (500 ca): OK")

# 12.6 từ cài sẵn "chắc chắn khớp" phải có mặt trong kết quả; từ "đột biến một vị trí" thường không khớp
rng = random.Random(6890)
for _ in range(300):
    L = rng.randint(2, 20)
    pattern = "".join(rng.choice(string.ascii_lowercase[:rng.randint(2, 10)]) for _ in range(L))
    perm = list(string.ascii_lowercase)
    rng.shuffle(perm)
    p = dict(zip(string.ascii_lowercase, perm))
    good = "".join(p[c] for c in pattern)
    res = Solution().findAndReplacePattern([good], pattern)
    assert res == [good]
    # đột biến: ép hai vị trí có chữ pattern khác nhau cùng một chữ từ -> mất đơn ánh
    pos = [i for i in range(L) for j in range(i + 1, L) if pattern[i] != pattern[j]]
    if pos:
        i = pos[0]
        j = next(j for j in range(i + 1, L) if pattern[i] != pattern[j])
        bad = list(good)
        bad[j] = bad[i]
        assert Solution().findAndReplacePattern(["".join(bad)], pattern) == []
print("12.6 từ chắc chắn khớp / từ bị phá đơn ánh: OK")

# 12.7 tính chất: kết quả bất biến khi áp dụng cùng một hoán vị lên mọi từ và pattern; và khi đảo thứ tự words
rng = random.Random(7890)
for _ in range(300):
    L = rng.randint(1, 8)
    alphabet = string.ascii_lowercase[:rng.randint(1, 5)]
    pattern = "".join(rng.choice(alphabet) for _ in range(L))
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 8))]
    perm = list(string.ascii_lowercase)
    rng.shuffle(perm)
    p = dict(zip(string.ascii_lowercase, perm))
    mapped_words = ["".join(p[c] for c in w) for w in words]
    mapped_pattern = "".join(p[c] for c in pattern)
    base = Solution().findAndReplacePattern(words, pattern)
    assert [p_ for p_ in map(lambda w: "".join(p[c] for c in w), base)] == Solution().findAndReplacePattern(mapped_words, mapped_pattern)
    assert sorted(Solution().findAndReplacePattern(words[::-1], pattern)) == sorted(base)
print("12.7 tính chất bất biến: OK")

12.4 ngẫu nhiên bảng chữ cái nhỏ (800 ca): OK, tổng 1212 từ khớp


12.5 ngẫu nhiên đúng ràng buộc (500 ca): OK
12.6 từ chắc chắn khớp / từ bị phá đơn ánh: OK
12.7 tính chất bất biến: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra ánh xạ **một chiều**

Ánh xạ xuôi nhất quán chưa đủ: hai chữ khác nhau của `pattern` có thể cùng ảnh (không phải hoán vị).

In [23]:
words, pattern = counters["chỉ ánh xạ xuôi"]
print("words =", words, "pattern =", pattern, "| sai ->", wrong_forward_only(words, pattern), "| đúng ->", Solution().findAndReplacePattern(words, pattern))
assert sorted(wrong_forward_only(words, pattern)) != sorted(Solution().findAndReplacePattern(words, pattern))
print("ví dụ 1:", wrong_forward_only(*examples[0][:2]), "(sai: có thêm 'ccc') so với", Solution().findAndReplacePattern(*examples[0][:2]))

words = ['baa', 'adc', 'abc', 'cba'] pattern = cad | sai -> ['baa', 'adc', 'abc', 'cba'] | đúng -> ['adc', 'abc', 'cba']
ví dụ 1: ['mee', 'aqq', 'ccc'] (sai: có thêm 'ccc') so với ['mee', 'aqq']


Vì sao sai: `"ccc"` thỏa "mỗi chữ của `pattern` có một ảnh" nhưng `a` và `b` cùng ảnh `c`. **Đúng:** kiểm tra thêm chiều ngược `word -> pattern` (hoặc đếm `set(zip(...))`).

### Lỗi 2 — Chỉ so số chữ khác nhau

Cùng số chữ khác nhau vẫn có thể có **hình dạng** khác.

In [24]:
words, pattern = counters["chỉ đếm số chữ khác nhau"]
print("words =", words, "pattern =", pattern, "| sai ->", wrong_distinct_count(words, pattern), "| đúng ->", Solution().findAndReplacePattern(words, pattern))
assert sorted(wrong_distinct_count(words, pattern)) != sorted(Solution().findAndReplacePattern(words, pattern))

words = ['dbbc', 'caaa', 'dcdc', 'bbad'] pattern = bddc | sai -> ['dbbc', 'bbad'] | đúng -> ['dbbc']


Vì sao sai: `"aab"` và `"abb"` đều có 2 chữ khác nhau nhưng vị trí lặp lại khác nhau. **Đúng:** so **hình dạng** (chuẩn hóa) hoặc ánh xạ theo từng vị trí.

### Lỗi 3 — So phân bố tần suất

Hai chuỗi có cùng đa tập số lần xuất hiện vẫn có thể khác nhau về **vị trí**.

In [25]:
words, pattern = counters["so phân bố tần suất"]
print("words =", words, "pattern =", pattern, "| sai ->", wrong_sorted_counts(words, pattern), "| đúng ->", Solution().findAndReplacePattern(words, pattern))
assert sorted(wrong_sorted_counts(words, pattern)) != sorted(Solution().findAndReplacePattern(words, pattern))

words = ['dbbc', 'caaa', 'dcdc', 'bbad'] pattern = bddc | sai -> ['dbbc', 'bbad'] | đúng -> ['dbbc']


Vì sao sai: `"abb"` và `"aab"` có cùng tần suất `{1, 2}` nhưng khác hình dạng. **Đúng:** ánh xạ phải theo **vị trí** `pattern[i] -> word[i]`.

### Lỗi 4 — Dùng chung bảng ánh xạ giữa các từ

Nếu khai báo `m1, m2` **bên ngoài** vòng lặp qua các từ thì ánh xạ của từ trước làm hỏng từ sau.

In [26]:
words, pattern = counters["dùng chung bảng giữa các từ"]
print("words =", words, "pattern =", pattern, "| sai ->", wrong_shared_map(words, pattern), "| đúng ->", Solution().findAndReplacePattern(words, pattern))
assert sorted(wrong_shared_map(words, pattern)) != sorted(Solution().findAndReplacePattern(words, pattern))

words = ['baa', 'adc', 'abc', 'cba'] pattern = cad | sai -> [] | đúng -> ['adc', 'abc', 'cba']


Vì sao sai: từ thứ hai bị so với ánh xạ đã ghi từ từ thứ nhất. **Đúng:** tạo bảng **mới** cho **mỗi** từ (đặt `m1, m2 = {}, {}` bên trong hàm `match(word)`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "thay thế chữ cái bằng hoán vị / đẳng cấu chuỗi / cùng cấu trúc lặp" ⇒ **kiểm tra song ánh** hoặc **chuẩn hóa theo lần xuất hiện đầu**.
- **Chuẩn hóa chuỗi:** `m.setdefault(c, len(m))` đánh số chữ theo lần xuất hiện đầu; hai chuỗi đẳng cấu ⇔ dạng chuẩn hóa bằng nhau.
- **Một dòng:** `len(set(a)) == len(set(b)) == len(set(zip(a, b)))` (hai chuỗi cùng độ dài).
- Luôn kiểm tra **hai chiều** (nhất quán và đơn ánh).
- Kết quả bất kỳ thứ tự: so sánh bằng `sorted`.
- Cùng họ: Isomorphic Strings (205), Word Pattern (290), Group Anagrams (49), Group Shifted Strings (249), Find Duplicate Subtrees (652, dùng chuẩn hóa cây).
- Test nên có: độ dài 1, `pattern` toàn giống/toàn khác nhau, hoán đổi hai chữ, từ trùng lặp, không từ nào khớp.

## 15. Biến thể

1. **Nhóm các từ đẳng cấu với nhau** (bài tập 5): nhóm theo dạng chuẩn hóa.
2. **Trả về hoán vị `p`** (hoặc ánh xạ) cho từng từ khớp.
3. **Isomorphic Strings (205)** với hai chuỗi: chính là `match_zip`.
4. Bài liên quan: Word Pattern (290), Group Shifted Strings (249).

Dưới đây là biến thể 2 (trả về ánh xạ) kiểm bằng hàm kiểm tra.

In [27]:
def mapping_for(word: str, pattern: str):
    """Trả về hoán vị đầy đủ của bảng chữ cái (dict 26 chữ) hoặc None nếu không khớp."""
    m1, m2 = {}, {}
    for x, y in zip(pattern, word):
        if m1.get(x, y) != y or m2.get(y, x) != x:
            return None
        m1[x], m2[y] = y, x
    rest_src = [c for c in string.ascii_lowercase if c not in m1]
    rest_dst = [c for c in string.ascii_lowercase if c not in m2]
    m1.update(zip(rest_src, rest_dst))
    return m1

rng = random.Random(15)
for _ in range(500):
    L = rng.randint(1, 8)
    alphabet = string.ascii_lowercase[:rng.randint(1, 5)]
    pattern = "".join(rng.choice(alphabet) for _ in range(L))
    word = "".join(rng.choice(alphabet) for _ in range(L))
    p = mapping_for(word, pattern)
    assert (p is not None) == oracle_relation(word, pattern)
    if p is not None:
        assert sorted(p.values()) == list(string.ascii_lowercase)          # đúng là hoán vị
        assert "".join(p[c] for c in pattern) == word
print(mapping_for("mee", "abb")["a"], mapping_for("mee", "abb")["b"], mapping_for("ccc", "abb"))

m e None


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị | Định nghĩa | `O(w k! L)` | `O(k)` | Oracle, bảng chữ cái nhỏ |
| 2. Hai bảng ánh xạ | Kiểm tra xuôi và ngược | `O(w L)` | `O(1)` | Trả ánh xạ, thoát sớm |
| 3. Chuẩn hóa | So hình dạng lặp lại | `O(w L)` | `O(L)` | **Cách nộp chính**, mở rộng sang nhóm |
| 4. `set(zip)` | Đếm ba tập | `O(w L)` | `O(L)` | Gọn một dòng |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(canon("abb"), canon("mee"), canon("ccc"))
print(len(set(zip("abb", "mee"))))
print(match_zip("dkd", "abb"))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `pattern = "abca"` và `words = ["xyzx", "xyzy", "pqrp", "aaaa"]`: ghi các từ khớp.

In [29]:
# Bài 2: điền kết quả tính tay
hand_matches = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm kiểm tra bằng hai bảng ánh xạ.

In [30]:
def exercise_3_match(word: str, pattern: str) -> bool:
    m1, m2 = {}, {}
    for x, y in zip(pattern, word):
        # TODO 1: nếu x đã có ảnh khác y, trả False
        # TODO 2: nếu y đã có tiền ảnh khác x, trả False
        # TODO 3: ghi nhận m1[x] = y và m2[y] = x
        pass
    return True


# assert exercise_3_match("mee", "abb") is True
# assert exercise_3_match("ccc", "abb") is False
# assert exercise_3_match("dkd", "abb") is False

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_zip(words, pattern)` bằng `set(zip(...))`.

In [31]:
def exercise_4_zip(words: List[str], pattern: str) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_4_zip(["abc", "deq", "mee", "aqq", "dkd", "ccc"], "abb")) == ["aqq", "mee"]
# assert sorted(exercise_4_zip(["a", "b", "c"], "a")) == ["a", "b", "c"]

### Bài 5 — Mini challenge: nhóm các từ đẳng cấu (khó)

Cho `words` (độ dài tùy ý), hãy viết `exercise_5_groups(words)` trả về **số nhóm** sao cho hai từ cùng nhóm khi và chỉ khi chúng **đẳng cấu** (cùng độ dài và có song ánh chữ cái biến từ này thành từ kia).

```python
exercise_5_groups(["abb", "mee", "aqq", "abc", "ccc", "xx", "yy", "xy"]) -> 5
exercise_5_groups(["a"])                                              -> 1
exercise_5_groups([])                                                 -> 0
```
(Các nhóm: `{abb, mee, aqq}`, `{abc}`, `{ccc}`, `{xx, yy}`, `{xy}`.)

In [32]:
def exercise_5_groups(words: List[str]) -> int:
    # TODO
    pass


# assert exercise_5_groups(["abb", "mee", "aqq", "abc", "ccc", "xx", "yy", "xy"]) == 5
# assert exercise_5_groups(["a"]) == 1
# assert exercise_5_groups([]) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `canon` đánh số chữ theo lần xuất hiện đầu.
- Mức 2: `"abb"` và `"mee"` đều có dạng `(0, 1, 1)`; các cặp của `zip("abb", "mee")` là `(a,m), (b,e), (b,e)`.
- Mức 3: `(0, 1, 1) (0, 1, 1) (0, 0, 0)`, `2`, `False`.

**Bài 2**
- Mức 1: `"abca"` có hình dạng `(0, 1, 2, 0)`: vị trí đầu và cuối bằng nhau, ba chữ đầu khác nhau.
- Mức 2: `"xyzx"` có `(0,1,2,0)`; `"xyzy"` có `(0,1,2,1)`; `"pqrp"` có `(0,1,2,0)`; `"aaaa"` có `(0,0,0,0)`.
- Mức 3: `["xyzx", "pqrp"]`.

**Bài 3**
- Mức 1: so sánh với `m1.get(x, y)` và `m2.get(y, x)`: nếu chưa có thì mặc định bằng chính giá trị đang xét.
- Mức 2: `if m1.get(x, y) != y or m2.get(y, x) != x: return False`.
- Mức 3: `m1[x], m2[y] = y, x`.

**Bài 4**
- Mức 1: `n = len(set(pattern))`.
- Mức 2: từ khớp nếu `len(set(w)) == n == len(set(zip(pattern, w)))`.
- Mức 3: `return [w for w in words if len(set(w)) == n == len(set(zip(pattern, w)))]`.

**Bài 5**
- Mức 1: hai từ đẳng cấu ⇔ dạng chuẩn hóa bằng nhau (độ dài tự nhiên khác nhau thì dạng khác nhau).
- Mức 2: dùng `set` các dạng chuẩn hóa (tuple).
- Mức 3: `len({canon(w) for w in words})`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(canon("abb"), canon("mee"), canon("ccc"))
print(len(set(zip("abb", "mee"))))
print(match_zip("dkd", "abb"))

assert (canon("abb"), canon("mee"), canon("ccc")) == ((0, 1, 1), (0, 1, 1), (0, 0, 0))
assert len(set(zip("abb", "mee"))) == 2
assert match_zip("dkd", "abb") is False

(0, 1, 1) (0, 1, 1) (0, 0, 0)
2
False


### Đáp án Bài 2

`"xyzx"` và `"pqrp"` có hình dạng `(0, 1, 2, 0)` như `"abca"`.

In [34]:
hand_matches_solution = ["xyzx", "pqrp"]
assert sorted(Solution().findAndReplacePattern(["xyzx", "xyzy", "pqrp", "aaaa"], "abca")) == sorted(hand_matches_solution)
assert oracle_find(["xyzx", "xyzy", "pqrp", "aaaa"], "abca") == sorted(hand_matches_solution)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_match_solution(word: str, pattern: str) -> bool:
    m1, m2 = {}, {}
    for x, y in zip(pattern, word):
        if m1.get(x, y) != y or m2.get(y, x) != x:
            return False
        m1[x], m2[y] = y, x
    return True

assert exercise_3_match_solution("mee", "abb") is True
assert exercise_3_match_solution("ccc", "abb") is False
assert exercise_3_match_solution("dkd", "abb") is False
rng = random.Random(3)
for _ in range(500):
    L = rng.randint(1, 7)
    alphabet = string.ascii_lowercase[:rng.randint(1, 5)]
    pattern = "".join(rng.choice(alphabet) for _ in range(L))
    word = "".join(rng.choice(alphabet) for _ in range(L))
    assert exercise_3_match_solution(word, pattern) == oracle_relation(word, pattern)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_zip_solution(words: List[str], pattern: str) -> List[str]:
    n = len(set(pattern))
    return [w for w in words if len(set(w)) == n == len(set(zip(pattern, w)))]

assert sorted(exercise_4_zip_solution(["abc", "deq", "mee", "aqq", "dkd", "ccc"], "abb")) == ["aqq", "mee"]
assert sorted(exercise_4_zip_solution(["a", "b", "c"], "a")) == ["a", "b", "c"]
rng = random.Random(4)
for _ in range(300):
    L = rng.randint(1, 7)
    alphabet = string.ascii_lowercase[:rng.randint(1, 5)]
    pattern = "".join(rng.choice(alphabet) for _ in range(L))
    words = ["".join(rng.choice(alphabet) for _ in range(L)) for _ in range(rng.randint(1, 6))]
    assert sorted(exercise_4_zip_solution(words, pattern)) == oracle_find(words, pattern)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_groups_solution(words: List[str]) -> int:
    return len({canon(w) for w in words})

def groups_oracle(words: List[str]) -> int:
    """Đếm số lớp tương đương bằng so sánh từng cặp (oracle quan hệ), độc lập với chuẩn hóa."""
    reps: List[str] = []
    for w in words:
        if not any(len(w) == len(r) and oracle_relation(w, r) for r in reps):
            reps.append(w)
    return len(reps)

assert exercise_5_groups_solution(["abb", "mee", "aqq", "abc", "ccc", "xx", "yy", "xy"]) == 5
assert exercise_5_groups_solution(["a"]) == 1
assert exercise_5_groups_solution([]) == 0
rng = random.Random(5)
for _ in range(400):
    alphabet = string.ascii_lowercase[:rng.randint(1, 4)]
    words = ["".join(rng.choice(alphabet) for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(0, 10))]
    assert exercise_5_groups_solution(words) == groups_oracle(words), words
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (chuẩn hóa theo lần xuất hiện đầu)

```python
class Solution:
    def findAndReplacePattern(self, words: List[str], pattern: str) -> List[str]:
        def f(s: str) -> List[int]:
            m = {}
            return [m.setdefault(c, len(m)) for c in s]   # đánh số chữ theo lần xuất hiện đầu

        target = f(pattern)
        return [w for w in words if f(w) == target]       # cùng hình dạng lặp lại <=> khớp song ánh
```

Công thức nhớ nhanh: khớp ⇔ `canon(word) == canon(pattern)` ⇔ `len(set(p)) == len(set(w)) == len(set(zip(p, w)))`.

### Checklist trước khi submit

- [ ] Kiểm tra **cả hai chiều** (nhất quán và đơn ánh), không chỉ ánh xạ xuôi?
- [ ] Bảng ánh xạ/chuẩn hóa tạo **mới cho mỗi từ**?
- [ ] Không so sánh phân bố tần suất hay số chữ khác nhau thay cho hình dạng?
- [ ] Kết quả được kiểm thử bằng `sorted` (thứ tự bất kỳ), giữ từ trùng lặp?
- [ ] Đã thử: độ dài 1, `pattern` toàn giống/toàn khác nhau, hoán đổi hai chữ, độ dài 20, không từ nào khớp?
- [ ] Tên method đúng `findAndReplacePattern(self, words, pattern)` và trả `List[str]`?